<a href="https://colab.research.google.com/github/Innovatewithapple/transformer-paper-rag/blob/main/VisionBasedRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers pdf2image faiss-cpu

In [ ]:

!apt-get install -y poppler-utils

In [10]:
import torch
import faiss
from pdf2image import convert_from_path
import numpy as np
from PIL import Image
from IPython.display import Image, display
import os
from transformers import ColQwen2ForRetrieval, ColQwen2Processor
from transformers.utils.import_utils import is_flash_attn_2_available
import base64
import requests
from google.colab import userdata

In [ ]:
#------Encoder-Model-------!
encoder_model = ColQwen2ForRetrieval.from_pretrained('vidore/colqwen2-v1.0-hf',device_map='auto',attn_implementation='flash_attention_2' if is_flash_attn_2_available() else 'sdpa')
processor = ColQwen2Processor.from_pretrained('vidore/colqwen2-v1.0-hf')

In [ ]:
#----Decoder-Model-------!

In [ ]:
#----paths-----!
pdf_path = '/content/NIPS-2017-attention-is-all-you-need-Paper.pdf'
output_folder = 'extracted_pdf_pages'

In [ ]:
def page_to_image(pdf_path,dpi,fmt,output_folder):
  #-----Create directory if not exist----!
  if not os.path.exists(output_folder):
    os.makedirs(output_folder)

  #-----convert pages to image-----!
  images = convert_from_path(pdf_path=pdf_path,dpi=dpi,output_folder=output_folder,fmt=fmt,transparent=False)
  return images

In [ ]:
images = page_to_image(pdf_path=pdf_path,dpi=150,fmt='png',output_folder=output_folder)

In [ ]:
display(Image(filename='/content/extracted_pdf_pages/9065f9b3-8d73-486a-ab46-cbbdbcae43a8-01.png', width=500))

In [12]:
def process_embedImages(pages,model,proc,batch_size=2) -> list[torch.Tensor]:
  all_embeddings = []

  for i in range(0,len(pages),batch_size):
    batch = pages[i:i+batch_size]
    inputs = proc(images=batch).to(model.device)

    with torch.no_grad():
      embedding = model(**inputs).embeddings

    print(f'Embedding Shape: {embedding.shape}')

    #Move to cpu
    for emb in embedding:
      all_embeddings.append(emb.cpu().detach())
  print(len(all_embeddings))
  return all_embeddings

In [ ]:
%%time
embedded = process_embedImages(pages=images,model=encoder_model,proc=processor,batch_size=2)

In [ ]:
page_embedded = torch.stack([emb.mean(dim=0) for emb in embedded])
page_embedded.shape

In [ ]:
type(page_embedded)

In [ ]:
#-----Faiss-----!
page_embedded_np = page_embedded.float().numpy()

faiss.normalize_L2(page_embedded_np)

dim = page_embedded_np.shape[1]

index = faiss.IndexFlatIP(dim)

index.add(page_embedded_np)

In [ ]:
def retrieve(query,top_k=3):
  input_text = processor(text=[query]).to(encoder_model.device)

  with torch.no_grad():
    embeddings = encoder_model(**input_text).embeddings

  query_embedding = embeddings.mean(dim=1) # here dim 1 use because query is token based. so we get average based on token
  query_np = (query_embedding.float().cpu().numpy())
  faiss.normalize_L2(query_np)

  scores,indices = index.search(query_np,top_k)
  results = []

  for score, idx in zip(scores[0], indices[0]):
      results.append({
            "page_index": int(idx),
            "score": float(score)
        })

  return results

In [ ]:
results = retrieve(query=questions[0])
results

In [ ]:
questions = [
    "What model architecture is introduced in the paper?",
    "What does the encoder in Transformer consist of?",
    "How many attention heads did they use?",
    "What BLEU score did they achieve on English-German translation?",
    "How long did training take on 8 GPUs?",
    "What is the key advantage over RNN models?",
    "What optimizer did they use?",
    "What is multi-head attention?"
]

# **LLM**

In [11]:
def RetriveAnswer_From_LLM(image,systemPrompt,userPrompt,API_KEY,url):
  #-----Header----@
  headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
  }

  #--Convert image to base64--!
  with open(image,"rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

  #-----payload-----@
  payload = {
      "messages":[
          {
              "role":"system","content":f"{systemPrompt}"
          },
          {
              "role":"user","contnet":[
                  {
                      "type":"image_url",
                      "image_url": {
                          "url": f"data:image/png;base64{image_b64}"
                      }
                  },
                  {
                      "type":"text",
                      "text":userPrompt
                  }
              ]
          }
      ],
      "max_tokens":80,
      "temprature":0.1,
      "stream":False
  }

  #-----Get Response----!
  request = requests.post(url=url,headers=headers,json=payload)
  return request


In [ ]:
nvidia_api_key = userdata.get('NVIDIA_API_KEY')
retrieved_image = images[results[0]['page_index']]
Answer = RetriveAnswer_From_LLM(image=retrieved_image,
                                systemPrompt=system_prompt,
                                userPrompt=user_prompt,
                                API_KEY=nvidia_api_key,
                                url="https://ai.api.nvidia.com/v1/vlm/google/paligemma")
print(Answer)


In [ ]:
system_prompt = "You are a research paper question-answering assistant."
user_prompt = f"""Answer the question using only the information visible in the provided document page.
Question:
{questions[0]}

If the answer is not visible on this page, say:
'I cannot find the answer on the provided docs.'
"""